## libreries

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
import torch
import pandas as pd
import os
import re

In [ ]:
print("CUDA disponible:", torch.cuda.is_available())
print("Nombre GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No detectada")

In [ ]:
device = 0 if torch.cuda.is_available() else -1
print("Usando dispositivo:" "GPU" if device == 0 else "CPU")

## modelo NER

In [ ]:
model_name = "Davlan/bert-base-multilingual-cased-ner-hrl"
# carga pipeline
ner_pipe = pipeline("ner", model=model_name, tokenizer=model_name, aggregation_strategy="simple", device = device)

##  Function to read and cut the final  document

In [ ]:
def load_tail(filepath, lines=40):
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read().splitlines()
         # --- NUEVO: unir líneas cortadas por guion o por salto ---
    clean_lines = []
    buffer = ""
    for line in text[-lines:]:
        line = line.strip()
        if not line:
            continue
        # Si la línea termina en guion, asumir que se cortó
        if line.endswith("-"):
            buffer += line[:-1]
        else:
            buffer += line + " "
            clean_lines.append(buffer.strip())
            buffer = ""

    tail_text = "\n".join(clean_lines)
    return tail_text

## excluir palabras:

In [ ]:
palabras_excluidas = set([
    "Presidente", "República", "Secretario", "Gobierno", "Senado", "Representantes",
    "Cámara", "Ejecutese", "publíquese", "Bogotá", "L", "S", "Secretarlo de", "Senador Secret", "El Senador Secretarin", "	Poder Ejecuiivo", "Senador Secretarin"
])

## ruta

In [ ]:
txt_path = "LEY-0001-1851plaintext.txt"
doc_tail = load_tail(txt_path)

## Detectar nombres

In [ ]:
entities = ner_pipe(doc_tail)
person_names = [ent["word"] for ent in entities if ent["entity_group"] == "PER"]

In [ ]:
# Filtro más fino: solo aceptar nombres con al menos 2 palabras y sin términos excluidos
def es_nombre_valido(nombre):
    palabras = nombre.split()
    if len(palabras) < 2:
        return False
    if any(palabra.strip(".,-()") in palabras_excluidas for palabra in palabras):
        return False
    return True

# Limpiar, quitar duplicados y filtrar
nombres_filtrados = list(set(
    name.strip().replace("\n", " ")
    for name in person_names
    if es_nombre_valido(name)
))

# Formato final
nombres_final = ", ".join(nombres_filtrados)

## Limpieza basica y eliminacion de duplicados

In [ ]:
clean_names = list(set([name.replace("\n", " ").strip() for name in person_names]))
print("Nombres detectados: ", clean_names)

## Uses


In [ ]:
# Nombre del archivo (extraído del path)
nombre_archivo = os.path.basename(txt_path)

# Unir los nombres separados por coma
nombres_unidos = ", ".join(clean_names)

df = pd.DataFrame([{
    "nombre_archivo": nombre_archivo,
    "nombres": nombres_final
}])
df.to_csv("firmantes_filtrados.csv", index=False, encoding="utf-8")

# Guardar a CSV
output_csv = "firmantes_extraidos.csv"

print(f"✅ Archivo CSV guardado como: {output_csv}")